# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [7]:
df = pd.read_csv("AviationData.csv", encoding='latin-1', low_memory=False)
df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [8]:
df.isna().sum()

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

In [10]:
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [11]:
columns_to_inspect = ['Event.Date', 'Amateur.Built', 'Make', 'Model']

# Check data types and look for missing values
print("--- Column Info ---")
df[columns_to_inspect].info()

print("\n--- Missing Values (NaN) ---")
print(df[columns_to_inspect].isnull().sum())

--- Column Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Event.Date     88889 non-null  object
 1   Amateur.Built  88787 non-null  object
 2   Make           88826 non-null  object
 3   Model          88797 non-null  object
dtypes: object(4)
memory usage: 2.7+ MB

--- Missing Values (NaN) ---
Event.Date         0
Amateur.Built    102
Make              63
Model             92
dtype: int64


In [12]:
df_filtered = df.dropna(subset=['Make', 'Model']).copy()

# Standardize text to uppercase and strip whitespace to prevent duplicates
df_filtered['Make'] = df_filtered['Make'].astype(str).str.upper().str.strip()
df_filtered['Model'] = df_filtered['Model'].astype(str).str.upper().str.strip()

In [13]:
# Convert 'Event.Date' to datetime objects
df_filtered['Event.Date'] = pd.to_datetime(df_filtered['Event.Date'], errors='coerce')

# cast to string and uppercase to catch variations like 'No' and 'NO'
df_filtered = df_filtered[df_filtered['Amateur.Built'].astype(str).str.upper() == 'NO']

# Keep only accidents from 1983 onwards
df_filtered = df_filtered[df_filtered['Event.Date'].dt.year >= 1983]

# Verify the final size of the dataset
print(f"Original dataset rows: {df.shape[0]}")
print(f"Filtered dataset rows: {df_filtered.shape[0]}")

Original dataset rows: 88889
Filtered dataset rows: 76888


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [14]:
injury_cols = [
    'Total.Fatal.Injuries', 
    'Total.Serious.Injuries', 
    'Total.Minor.Injuries', 
    'Total.Uninjured'
]

# 2. Impute missing values with 0
df_filtered[injury_cols] = df_filtered[injury_cols].fillna(0)

# 3. Construct 'Total_Passengers' by summing across the injury categories (axis=1)
df_filtered['Total_Passengers'] = df_filtered[injury_cols].sum(axis=1)

In [15]:
# 4. Construct metric for fatal/serious injuries ('Severe_Injury_Rate')
fatal_serious_sum = df_filtered['Total.Fatal.Injuries'] + df_filtered['Total.Serious.Injuries']

# Divide by total passengers, and fill any resulting NaNs (from dividing by 0) with 0
df_filtered['Severe_Injury_Rate'] = (fatal_serious_sum / df_filtered['Total_Passengers']).fillna(0)

# 5. Construct metric for robustness to destruction ('Is_Destroyed')
# Creates a 1 if the damage string matches 'DESTROYED', otherwise 0
df_filtered['Is_Destroyed'] = df_filtered['Aircraft.damage'].apply(
    lambda x: 1 if str(x).strip().upper() == 'DESTROYED' else 0
)

In [16]:
# Inspect the final clean dataframe to verify the new columns
df_filtered[['Make', 'Model', 'Total_Passengers', 'Severe_Injury_Rate', 'Is_Destroyed']].head()

,Make,Model,Total_Passengers,Severe_Injury_Rate,Is_Destroyed
3600,PICCARD,AX-6,2.0,0.5,0
3601,CESSNA,182P,4.0,0.0,0
3602,CESSNA,182RG,2.0,0.0,0
3603,CESSNA,182P,1.0,0.0,0
3604,PIPER,PA-28R-200,2.0,0.0,0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [17]:
# 1. Inspect unique values to see what is going on with the 'Aircraft.damage' column
print("Original Damage Categories:", df_filtered['Aircraft.damage'].unique())

# 2. Impute missing values with 'UNKNOWN'
df_filtered['Aircraft.damage'] = df_filtered['Aircraft.damage'].fillna('UNKNOWN')

# 3. Standardize the text (uppercase and strip whitespace)
df_filtered['Aircraft.damage'] = df_filtered['Aircraft.damage'].astype(str).str.upper().str.strip()

# 4. Construct the derived column: 'Is_Destroyed'
# using a lambda function to check if the cleaned string equals 'DESTROYED'
df_filtered['Is_Destroyed'] = df_filtered['Aircraft.damage'].apply(
    lambda x: 1 if x == 'DESTROYED' else 0
)


Original Damage Categories: [nan 'Substantial' 'Destroyed' 'Minor' 'Unknown']


In [18]:
# Verify the cleaning and the new derived column
print("\n--- Value Counts for Derived 'Is_Destroyed' Column ---")
print(df_filtered['Is_Destroyed'].value_counts())

display(df_filtered[['Make', 'Model', 'Aircraft.damage', 'Is_Destroyed']].head())


--- Value Counts for Derived 'Is_Destroyed' Column ---
Is_Destroyed
0    61432
1    15456
Name: count, dtype: int64


,Make,Model,Aircraft.damage,Is_Destroyed
3600,PICCARD,AX-6,UNKNOWN,0
3601,CESSNA,182P,SUBSTANTIAL,0
3602,CESSNA,182RG,SUBSTANTIAL,0
3603,CESSNA,182P,SUBSTANTIAL,0
3604,PIPER,PA-28R-200,SUBSTANTIAL,0


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [19]:
# 1 & 2: Standardize text case and strip whitespace
df_filtered['Make'] = df_filtered['Make'].astype(str).str.upper().str.strip()

# Check how many unique Makes before filtering
print(f"Unique Makes before threshold filter: {df_filtered['Make'].nunique()}")


Unique Makes before threshold filter: 1874


In [20]:
# First, get the counts for each Make
make_counts = df_filtered['Make'].value_counts()
# Then, filter the DataFrame
df_filtered = df_filtered[df_filtered['Make'].isin(make_counts[make_counts >= 50].index)]

In [21]:
df_filtered['Make'].value_counts().head(10)

Make
CESSNA      25736
PIPER       14098
BEECH        5093
BOEING       2643
BELL         2594
MOONEY       1271
ROBINSON     1196
GRUMMAN      1064
BELLANCA      978
HUGHES        877
Name: count, dtype: int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [22]:
# 1. Get rid of any NaNs in the Model column
df_filtered = df_filtered.dropna(subset=['Model']).copy()

# Standardize the Model text (uppercase and strip spaces) to match our Make column
df_filtered['Model'] = df_filtered['Model'].astype(str).str.upper().str.strip()

# 2. Inspect: Are model labels unique to each make? 
# Let's count how many unique 'Makes' are associated with each 'Model'
model_make_counts = df_filtered.groupby('Model')['Make'].nunique()

# Filter to find models that are shared by more than 1 Make
shared_models = model_make_counts[model_make_counts > 1]

print(f"Number of Model names shared by multiple manufacturers: {len(shared_models)}")
if len(shared_models) > 0:
    print("This proves models are NOT unique. We must create a combined identifier.\n")

Number of Model names shared by multiple manufacturers: 466
This proves models are NOT unique. We must create a combined identifier.



In [23]:
# 3. Create a unique identifier for a given plane type
df_filtered['Plane_Type'] = df_filtered['Make'] + ' - ' + df_filtered['Model']

# Inspect the new column and see the most common unique planes
display(df_filtered[['Make', 'Model', 'Plane_Type']].head())

,Make,Model,Plane_Type
3601,CESSNA,182P,CESSNA - 182P
3602,CESSNA,182RG,CESSNA - 182RG
3603,CESSNA,182P,CESSNA - 182P
3604,PIPER,PA-28R-200,PIPER - PA-28R-200
3605,CESSNA,140,CESSNA - 140


In [24]:
print(df_filtered['Plane_Type'].value_counts().head(10))

Plane_Type
CESSNA - 152         2229
CESSNA - 172         1649
CESSNA - 172N        1093
PIPER - PA-28-140     863
CESSNA - 172M         759
CESSNA - 150          752
CESSNA - 172P         665
CESSNA - 182          617
CESSNA - 180          595
CESSNA - 150M         551
Name: count, dtype: int64


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [25]:
# Strip whitespace and convert to Title Case. 
df_filtered['Engine.Type'] = df_filtered['Engine.Type'].str.strip().str.title()
# Standardize various unknown representations to a single string
df_filtered['Engine.Type'] = df_filtered['Engine.Type'].replace({'Unk': 'Unknown', 'None': 'Unknown'})

# Strip whitespace and convert to UPPERCASE
df_filtered['Weather.Condition'] = df_filtered['Weather.Condition'].str.strip().str.upper()
# Standardize unknowns
df_filtered['Weather.Condition'] = df_filtered['Weather.Condition'].replace({'UNKNOWN': 'UNK'})
df_filtered['Number.of.Engines'] = pd.to_numeric(df_filtered['Number.of.Engines'], errors='coerce')

In [26]:
# Strip whitespace and convert to Title Case
df_filtered['Purpose.of.flight'] = df_filtered['Purpose.of.flight'].str.strip().str.title()

# Strip whitespace and convert to Title Case
df_filtered['Broad.phase.of.flight'] = df_filtered['Broad.phase.of.flight'].str.strip().str.title()
df_filtered['Broad.phase.of.flight'] = df_filtered['Broad.phase.of.flight'].replace({'Unk': 'Unknown'})

print(df_filtered[['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 
          'Purpose.of.flight', 'Broad.phase.of.flight']].dtypes)

Engine.Type               object
Weather.Condition         object
Number.of.Engines        float64
Purpose.of.flight         object
Broad.phase.of.flight     object
dtype: object


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [27]:
missing_percentages = df_filtered.isnull().mean() * 100

print("Percentage of missing data per column:")
print(missing_percentages[missing_percentages > 0].sort_values(ascending=False))


Percentage of missing data per column:
Schedule                 84.649123
Air.carrier              81.177255
FAR.Description          69.709873
Aircraft.Category        69.435928
Longitude                63.167831
Latitude                 63.160734
Airport.Code             43.494862
Airport.Name             40.867541
Broad.phase.of.flight    28.958724
Publication.Date         17.299722
Engine.Type               7.688923
Purpose.of.flight         7.562596
Report.Status             7.170840
Number.of.Engines         6.842957
Weather.Condition         5.250383
Registration.Number       1.549991
Injury.Severity           1.249077
Country                   0.275365
Location                  0.063873
dtype: float64


In [28]:
threshold = 60.0 

# Identify the names of the columns that exceed the threshold
columns_to_drop = missing_percentages[missing_percentages > threshold].index

# 3. Execute: Drop those columns from the dataframe
df = df_filtered.drop(columns=columns_to_drop)

In [29]:
print(list(columns_to_drop))

['Latitude', 'Longitude', 'Aircraft.Category', 'FAR.Description', 'Schedule', 'Air.carrier']


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [30]:
output_filename = "aviation_accidents_cleaned.csv"
df_filtered.to_csv(output_filename, index=False)

print(f"Successfully saved cleaned data to {output_filename}")

Successfully saved cleaned data to aviation_accidents_cleaned.csv
